# Milestone 3: Testing and Validation Baseline

This notebook validates the Milestone 2 severity-by-speed workflow. It reports data quality, tests summary consistency, and creates the original charts.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

root = Path.cwd().resolve()
if not (root / "src" / "crash_validation.py").exists():
    root = root.parent
assert (root / "src" / "crash_validation.py").exists(), "Open notebook from repository or notebooks folder"
sys.path.insert(0, str(root))
from src.crash_validation import (load_crash_data, prepare_crash_data, severity_counts, severity_by_speed, validate_totals)
print("Repository root:", root)


## 1. Run automated tests

The tests use a synthetic dataset with known expected answers, so the tests can run even without access to the real CSV.


In [ ]:
import subprocess
result = subprocess.run([sys.executable, "-m", "pytest", "-q", str(root / "tests")], cwd=root, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, "Some automated tests failed"


## 2. Validate real data

Check the path and inspect the data-quality report. Missing or invalid required values are excluded only from the summaries.


In [ ]:
DATA_PATH = root / "data" / "right_turn_crashes.csv"
df = load_crash_data(DATA_PATH)
clean, quality = prepare_crash_data(df)
print("Data quality report:", quality)
print("Raw columns:", df.columns.tolist())
print("Severity labels:", sorted(clean["CrashSeverity"].unique().tolist()))
print("Speed limits:", sorted(clean["SpeedLimit"].unique().tolist()))


## 3. Verify independent summary totals


In [ ]:
counts = severity_counts(clean)
by_speed = severity_by_speed(clean)
assert validate_totals(clean, counts, by_speed)
print("PASS: both summaries match the valid row count")
display(counts.rename("Crash count").to_frame())
display(by_speed)


## 4. Reproduce Milestone 2 charts using validated rows


In [ ]:
ax = counts.plot(kind="bar", figsize=(8, 5))
ax.set(xlabel="Crash severity", ylabel="Number of crashes", title="Validated crash severity distribution")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
if by_speed.empty:
    print("No valid observations available for the grouped chart")
else:
    ax = by_speed.plot(kind="bar", figsize=(10, 6))
    ax.set(xlabel="Speed limit (mph)", ylabel="Number of crashes", title="Validated crash severity by speed limit")
    plt.xticks(rotation=0)
    plt.legend(title="Crash severity", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


## 5. Interpretation and limitations

- Tests verify known counts, data-quality rules, missing inputs, and internal consistency.
- Crash counts are not crash rates; exposure (e.g., AADT) is not incorporated.
- Zero speed limits and unfamiliar severity labels require manual review.
- Results are descriptive and do not demonstrate causation.
- Copy the actual test results and data-quality numbers into the submission summary after running this notebook.
